# HR Policy Assistant: a basic RAG agent

**RAG (Retrieval-Augmented Generation):** instead of relying only on what the model learned in training, you look up the relevant parts of *your own* documents and give them to the model along with the question. The model then answers from that text.

The notebook has two phases:

```
INDEXING (runs once)
HR Policy.txt → load → split into chunks → embed each chunk → store vectors in FAISS

ANSWERING (runs for every question)
question → Claude → search_hr_policy tool → FAISS finds the closest chunks → Claude answers from them
```

Steps 1–3 are setup, 4–7 are indexing, 8–9 are retrieval, and 10–17 are the model and the agent.

## 1. Imports and environment

**What:** Loads your API keys from `.env` and imports one class for each stage of the pipeline.

| Import | Stage |
|---|---|
| `TextLoader` | Load the file |
| `RecursiveCharacterTextSplitter` | Split it into chunks |
| `JinaEmbeddings` | Turn text into vectors |
| `FAISS` | Store and search the vectors |
| `ChatAnthropic` | The LLM (Claude) |
| `create_agent` | The loop that lets Claude call tools |

**Remember:**
- `load_dotenv()` copies the values in `.env` into environment variables, so keys stay out of your code. `.env` is listed in `.gitignore`, so it won't be committed.
- The `DeprecationWarning` about `langchain-community` is safe to ignore for now. The package still works; LangChain is moving these classes into standalone packages.

In [9]:
#Imports 

import os
from dotenv import load_dotenv

load_dotenv()

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

## 2. Read the API keys

**What:** Reads the Claude and Jina keys from the environment variables that `load_dotenv()` just set.

**Remember:**
- The names must match `.env` exactly (`CLAUDE_API_KEY`, `JINA_API_KEY`).
- `os.getenv` returns `None` for a missing name instead of raising an error. The error only shows up later, when a model tries to use the key, which makes typos confusing to track down.

In [10]:
claude_key = os.getenv('CLAUDE_API_KEY')
jina_key = os.getenv('JINA_API_KEY')

## 3. Point to the source document

**What:** Stores the path to the HR policy file.

**Remember:** `os.path.join` with a single argument returns the string unchanged. It's useful when you join parts, e.g. `os.path.join(folder, "HR Policy.txt")`.

In [11]:
DATA_FILE_PATH = os.path.join('/Users/raghavendra/Desktop/AI Engineering/BuildingRAG/HR Policy.txt')

## 4. Load the document

**What:** `TextLoader` reads the file and returns a list of LangChain `Document` objects. One `.txt` file gives one `Document`.

**A `Document` has two parts:**
- `page_content`: the text itself
- `metadata`: information about the text, here `{'source': <file path>}`

**Why it matters:** the metadata is copied onto every chunk in the next step, so after retrieval you can still tell where a chunk came from. That's how RAG apps cite sources or filter by document. Other loaders work the same way: for example, `PyPDFLoader` returns one `Document` per PDF page.

In [12]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load() # - This converts to a Langchain Document, it has two main components 1. PAGE CONTENT - the actual content of the Data Source, 2. MetaData - Additional Information about the Data. 
print(documents[0])
print(documents[0].page_content)
print(documents[0].metadata)

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leav

## 5. Split into chunks

**What:** Splits the document into smaller pieces. Each chunk gets its own embedding and is searched on its own.

**How the splitter works:** it tries to cut at blank lines (`"\n\n"`) first, then at single line breaks, then at spaces, then between characters. It packs whole pieces together until adding another would exceed `chunk_size`.

**Why 500 / 50 here:**
- `chunk_size` counts **characters, not tokens**. 500 characters is about 100–125 tokens.
- Every section of this file is under 500 characters (the largest is 451), and any two neighbouring sections together are over 500. So you get **9 chunks: the header plus one per policy section**. Each chunk is exactly one topic, which is ideal.
- `chunk_overlap = 50` has **no effect on this file**. Overlap is made of whole pieces from the end of the previous chunk, and every piece here (a whole section) is longer than 50 characters. Overlap matters for long text without paragraph breaks, where a cut can land mid-sentence.

**Trade-off:** chunks that are too small lose context ("is processed within 45 days": what is?). Chunks that are too large force one vector to represent several topics, so it matches many questions weakly.

**Remember:** `split_documents` keeps each chunk's `metadata`. Choose chunk settings by measuring retrieval on test questions, not by copying defaults.

In [13]:
#Chunking 

# Upon instantiating the RecursiveCharacterTextSplitter we're also 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, 
    chunk_overlap = 50
    ) 

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': '/Users/raghavendra/Desktop/AI Engineering/BuildingRAG/HR Policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': '/Users/raghavendra/Desktop/AI Engineering/BuildingRAG/HR Policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': '/Users/raghavendra/Desktop/AI Engineering/BuildingRAG/HR Policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully rem

## 6. Create the embedding model

**What:** Sets up the Jina model that turns text into an **embedding**: a list of numbers (1,024 for `jina-embeddings-v3`) where texts with similar meanings end up close together. This cell only configures the client; nothing is sent to Jina yet.

**Why:** vector search compares meaning, not exact words, so "sick days" can match "sick leave".

**Remember:**
- The **same** embedding model must embed both the chunks and the questions. Vectors from different models can't be compared.
- The embedding model and the LLM are separate choices: Jina only produces vectors, and Claude writes the answers. Anthropic doesn't offer an embedding model.

In [ ]:
#Embedding

embeddings_model = JinaEmbeddings(jina_api_key = jina_key, model_name="jina-embeddings-v3")


## 7. Embed the chunks and store them in FAISS

**What:** `FAISS.from_documents` sends all 9 chunks to Jina to be embedded (the first real Jina call), then builds a FAISS index: a structure that quickly finds the stored vectors closest to a new vector.

**Remember:**
- The index lives **in memory**. Restarting the kernel deletes it, and re-running this cell embeds everything again. For bigger datasets, save it with `vector_store.save_local("folder")` and reload it with `FAISS.load_local(...)`. Loading requires `allow_dangerous_deserialization=True` because it unpickles a file, so only load indexes you created yourself.
- By default this does **exact** search: it compares the query with every stored vector. That's fine for 9 chunks. With millions of vectors you'd switch to an approximate index (e.g. HNSW or IVF), which is much faster and slightly less exact.

In [15]:
# Storing the Vector Embeddings 

vector_store = FAISS.from_documents(chunks, embeddings_model)

## 8. Test retrieval on its own

**What:** Embeds the test question with the same Jina model and returns the `k = 2` chunks whose vectors are closest to it.

**Why test this separately:** if retrieval brings back the wrong chunks, Claude can't give the right answer. Checking retrieval on its own tells you which half of the pipeline is failing.

**Remember:**
- Search **always returns k results**, relevant or not. Match 2 (HOLIDAYS) has nothing to do with sick leave; it's simply the next closest chunk.
- Use `vector_store.similarity_search_with_score(...)` to see the distances. With this FAISS index, a **lower** score means a closer match.

In [16]:
# Peforming Similarity Search

test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query, k = 2)

for i, match in enumerate(top_matches, start = 1):
    print(f'Match {i}: {match.page_content}', end = "\n\n")


Match 1: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

Match 2: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



## 9. Wrap retrieval as a tool for the agent

**What:**
- `as_retriever` wraps the vector store in LangChain's standard retriever interface: `retriever.invoke(question)` returns the top `k = 3` chunks.
- `search_hr_policy` runs the retriever and joins the chunks into one string. That string is exactly what Claude receives when it uses the tool.

**Why the docstring and type hint matter:** `create_agent` turns this function into a tool. The **docstring becomes the tool description** that Claude reads to decide when to search, and `question: str` defines the tool's input. Write the docstring for the model, not just for yourself.

**Remember:** `k` is a trade-off. More chunks give a better chance that the answer is included, but also more irrelevant text and more tokens per question.

In [17]:
retriever = vector_store.as_retriever(search_kwargs = {"k": 3}) # returns the TOP 3 Revelant info from the DOC.

def search_hr_policy(question: str) -> str:
    """
    Search the HR Policy Document for information about Leave, Work from home, Probation, Notice Period, Reimbursement, Code of Conduct, Holidays, or Exit Process.
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

## 10. Configure Claude (the "generation" in RAG)

**What:** Creates the chat model that the agent will use.

| Setting | Meaning |
|---|---|
| `model` | Which Claude model to call |
| `api_key` | Your key from `.env` |
| `max_tokens` | Hard cap on response length. If Claude hits it, the answer is cut off and `stop_reason` is `"max_tokens"` |
| `effort` | How much Claude thinks before answering. Lower is faster and cheaper; higher is more thorough |
| `betas` + `model_kwargs` | Safety fallback: if Claude declines a request, the API retries it on another Claude model |

**Remember:**
- There's no `temperature`: Claude Opus 5.5 rejects `temperature`, `top_p` and `top_k`.
- LangChain chat models share one interface. Switching from Groq to Claude only needed this cell, the import and the key; the rest of the notebook stayed the same.

In [18]:
llm = ChatAnthropic(
    model = "claude-opus-5-5",
    api_key = claude_key,
    max_tokens = 16000, # upper limit on the length of each response
    effort = "medium",  # how much Claude thinks before answering: "low" | "medium" | "high" | "xhigh" | "max"
    # No temperature here - Claude Opus 5.5 doesn't accept temperature/top_p/top_k (the request fails if you set them).

    # If Claude declines a request for safety reasons, the API automatically retries it on a fallback Claude model.
    betas = ["server-side-fallback-2026-07-01"],
    model_kwargs = {"fallbacks": "default"},
)

## 11. Sanity check: call Claude directly

**What:** Sends one message straight to Claude, with no retrieval and no tools, to check that the key and model work.

**Remember:** this answer comes from what Claude learned in training, not from your HR document. The agent below looks up the document first; that difference is what RAG adds.

In [19]:
test_response = llm.invoke("Hey is learning RAG hard? Reply professionaly")

## 12. Read the answer text

**What:** Shows only the text of Claude's reply.

**Why `.text` and not `.content`:** the reply is an `AIMessage` whose `content` is a **list of blocks**: a `thinking` block (a short summary of Claude's reasoning) and a `text` block (the answer). `.text` joins just the text blocks.

**Remember:** `test_response.usage_metadata` shows the token counts, which is what you pay for.

In [20]:
# Claude returns a list of content blocks (a thinking block + a text block), so use .text to get just the answer
test_response.text

'Learning Retrieval-Augmented Generation (RAG) is **moderately challenging**, but very approachable if you have some programming experience. The difficulty depends on your background and how deep you want to go.\n\n### What Makes It Manageable\n- **The core concept is simple:** retrieve relevant information from a knowledge source, then pass it to a language model to generate a grounded answer.\n- **Mature tooling exists:** frameworks such as LangChain, LlamaIndex, and Haystack let you build a basic RAG pipeline in a few dozen lines of code.\n- **Abundant learning resources:** tutorials, documentation, and open-source examples are widely available.\n\n### What Makes It Challenging\n- **Retrieval quality:** choosing effective chunking strategies, embedding models, and similarity search methods takes experimentation.\n- **Vector databases:** understanding tools like FAISS, Pinecone, Chroma, or Weaviate, along with indexing and metadata filtering.\n- **Evaluation:** measuring whether answ

## 13. Build the agent

**What:** `create_agent` connects Claude, the tool and the system prompt into a loop:
1. Send the conversation and the tool description to Claude.
2. If Claude asks to use a tool, run it and send the result back.
3. Repeat until Claude answers without asking for a tool.

**Why an agent:** Claude decides *whether* to search and *what* to search for. It can rephrase the question or search more than once; for a two-part question it may run two searches at the same time. A simpler "RAG chain" always retrieves once, with the user's exact words, before answering.

**Remember:** the system prompt sets the rules. "Say you don't know" is your guard against made-up answers when retrieval misses.

In [21]:
# Creating an AI Agent

HR_Assistant  = create_agent(
    model = llm, 
    tools = [search_hr_policy], 
    system_prompt = """ 
    
    You are a friendly HR assistant. 
    Always use the search_hr_policy tool to look up for facts before answering. 
    If the answer isn't in the search results, say you don't know instead of re-inventing/guessing."

    """
)

## 14. Helper to ask the agent a question

**What:** Wraps `HR_Assistant.invoke` so you can ask a question in one line.

**Remember:**
- The input is a dict with a `messages` list. Each message has a `role` (`"user"`) and `content`.
- It returns the **whole conversation state** (every message), not just the answer, despite the docstring and the `-> str` hint. The cells below pull the answer out.
- There's **no memory between calls**: each call starts a fresh conversation. To keep history, you'd pass a `checkpointer` to `create_agent` and a `thread_id` with each call.

In [22]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG Agent and Print a nicely formatted answer"""
    # print("Question: ", question, end = "\n")

    response = HR_Assistant.invoke(
        {
            "messages": [
                {
                    'role': "user", 
                    "content": question
                }
            ]
        }
    )
    return response 


## 15. Ask the agent something

**What:** Sends a greeting to the agent.

**What to notice:** Claude answers without calling the tool, because a greeting has no facts to look up. Try an HR question such as `"How many sick days do I get?"` and you'll see the tool call in the output below.

In [23]:
response_output = ask_hr_assistant('Hello, How are you doing Today?')

## 16. Inspect the raw result

**What:** Prints everything the agent returned.

**How to read it:** the `messages` list is the whole conversation, in order:
- `HumanMessage`: your question
- `AIMessage`: Claude's reply. If it decided to search, `tool_calls` shows the tool name and the query it used
- `ToolMessage`: the chunks the tool returned (only present when it searched)
- the last `AIMessage`: the answer

**Also worth reading:** `usage_metadata` (token counts) and `response_metadata['stop_reason']` (`end_turn` means Claude finished normally).

In [24]:
print(response_output)


{'messages': [HumanMessage(content='Hello, How are you doing Today?', additional_kwargs={}, response_metadata={}, id='1db801aa-c144-4913-85a5-b4ef20f19fa8'), AIMessage(content=[{'signature': 'CAQSgAUKEAgSGAI4AUIIdGhpbmtpbmcSDHLew7Q2DWQ+asYLuhoM+YsRYACpjuTaa0m/IjCYHDZbzF9YIFK4ANlBEdu/+xnmCPu6EDfw3wgqqlxuj05Wi9Fa+44QkAc1zpbUiBIqnQTIKVKyifsosJQs6//qFosc553oViBXx2IqLGXxDbFzIE/eyh5LEsTYmqOPBp40uMnQQ+wwAqiBRF74a2F1lPG5MycEqlOcMOEY4bOp0kopROgpqRVhLnhpIQWwF6X8KYfcR/uFH6YCOWoxsi+nu4hCCkyGob0mfouO+eAFJVB9r2dVaQ8s0Xikk8gASa4NOmcU++DdTz2slQFOzM031j2ATzPqNqP/RH3GjDBWRdfDP+4v0HyEadYirJBBEIFqikTKCzfrM8zLqecdcaqu4knazBZ6zHdCYtyWlBCHBwFSLh69mL+8pJqz8Rf99XbnAVZAdrZ2IToeYbVFy08jrX2gtvmf7gbURSoVJmY49a3WAhxc7G7fZ8jcp40NiYyHp7/tFfgpyVJBNvxYzQrTe1uW+e0P7ymylMHsBjB5W1oLjHyQ43HpHnNlSexrAPUfZhjcHEWcQzdkTgHoqe4QlCI3bqr0aCZoIgHMKrx4JXp1Ypl5tHGCMPJR30NHMnYx3CPDNAz5JOI8tOi4m3sSIGgrtU4f+1arkCuiUG6mVLDWty+4THbtYY75IRUrYLDyFOMO/qMtVPygBSTS+Wpc/L/TPovUG0+IG2/dzMrD8iiMh16PpE3lZWkSouNLpGCDtK2oxpOiD3+LGO8jo+RejjtggDom18F2

## 17. Print the question and the answer

**What:** Pulls out the question (`messages[0]`) and the final answer (`messages[-1]`).

**Why `[-1]`:** when the agent searches, the tool call and the tool result sit between the question and the answer. The answer is always the **last** message, not necessarily the second.

In [25]:
# [-1] is the final answer. When the agent uses the tool, the list also holds the tool call and the tool result in between.
final_message = response_output['messages'][-1]

print(f"Human Question: {response_output['messages'][0].content}")
print(f"AI Response: {final_message.text}")
print(f"Full AI Message: {final_message}")

Human Question: Hello, How are you doing Today?
AI Response: Hi there! I'm doing well, thanks for asking. I hope your day is going well too.

I'm your HR assistant, and I can answer questions about company HR policies, including:

- 🌴 **Leave** (types of leave and how to apply)
- 🏠 **Work from home**
- 📋 **Probation**
- ⏳ **Notice period**
- 💰 **Reimbursements**
- 🤝 **Code of conduct**
- 🎉 **Holidays**
- 🚪 **Exit process**

What would you like to know?
Full AI Message: content=[{'signature': 'CAQSgAUKEAgSGAI4AUIIdGhpbmtpbmcSDHLew7Q2DWQ+asYLuhoM+YsRYACpjuTaa0m/IjCYHDZbzF9YIFK4ANlBEdu/+xnmCPu6EDfw3wgqqlxuj05Wi9Fa+44QkAc1zpbUiBIqnQTIKVKyifsosJQs6//qFosc553oViBXx2IqLGXxDbFzIE/eyh5LEsTYmqOPBp40uMnQQ+wwAqiBRF74a2F1lPG5MycEqlOcMOEY4bOp0kopROgpqRVhLnhpIQWwF6X8KYfcR/uFH6YCOWoxsi+nu4hCCkyGob0mfouO+eAFJVB9r2dVaQ8s0Xikk8gASa4NOmcU++DdTz2slQFOzM031j2ATzPqNqP/RH3GjDBWRdfDP+4v0HyEadYirJBBEIFqikTKCzfrM8zLqecdcaqu4knazBZ6zHdCYtyWlBCHBwFSLh69mL+8pJqz8Rf99XbnAVZAdrZ2IToeYbVFy08jrX2gtvmf7gbURSoVJmY49a3WAh